## Tylko testy do większych zbiorów danych

In [2]:
import torch

In [9]:
import os
import pandas as pd
from torchvision.io import read_image
import re
import wfdb
import wfdb.processing
import scipy
from torch.utils.data import Dataset
import numpy as np
import json
import torch.nn as nn
import torch
from tqdm import tqdm
import torch.nn.functional as F

In [29]:
import os
import re
import numpy as np
import torch
from torch.utils.data import Dataset
import wfdb
import wfdb.processing
import scipy.signal

def extract_qrs_segment(signal, qrs_index, N):
    """
    Ekstrahuje segment sygnału EKG wokół zadanego indeksu QRS.
    Dopełnia brakujące próbki medianą sygnału, jeśli QRS jest blisko granic.
    """
    start_idx = qrs_index - N
    end_idx = qrs_index + N + 1  # +1, bo Python używa wykluczającego zakresu
    
    if start_idx < 0:
        padding_left = np.median(signal[:end_idx])
        segment = np.concatenate([np.full(-start_idx, padding_left), signal[:end_idx]])
    elif end_idx > len(signal):
        padding_right = np.median(signal[start_idx:])
        segment = np.concatenate([signal[start_idx:], np.full(end_idx - len(signal), padding_right)])
    else:
        segment = signal[start_idx:end_idx]
    
    return segment

class SHDB_QRS(Dataset):
    def __init__(self, N, dataset_dir, fs=100):
        """
        N - liczba próbek przed i po załamku QRS w segmencie.
        dataset_dir - ścieżka do katalogu z danymi EKG.
        fs - docelowa częstotliwość próbkowania.
        """
        self.N = N
        self.fs = fs
        self.rr_segments = []
        self.labels = []

        exclusion_lst = []  # Lista plików do wykluczenia, jeśli potrzebna
        for file in os.listdir(dataset_dir):
            name = re.match(r'^(.*\d\d+)\.atr$', file)
            if name and name.group(1) not in exclusion_lst:
                print(f"Przetwarzanie: {name.group(1)}")
                record = wfdb.rdsamp(f"{dataset_dir}{name.group(1)}")
                # annotation = wfdb.rdann(f"{dataset_dir}{name.group(1)}", 'atr')
                annotation = wfdb.rdann(f"{dataset_dir}{name.group(1)}", 'atr', return_label_elements=['symbol'])
                signal = record[0][:, 0]
                fs_original = record[1]["fs"]
        
                # Filtracja sygnału
                cutOff = 20
                b, a = scipy.signal.butter(5, cutOff, fs=fs_original, btype='low', analog=False)
                filtered_signal = scipy.signal.lfilter(b, a, signal)

                # Resampling do docelowej częstotliwości próbkowania
                num_samples_target = int(filtered_signal.shape[0] * fs / fs_original)
                resampled_signal = scipy.signal.resample(filtered_signal, num_samples_target)
                resampled_annotations = (annotation.sample * fs) / fs_original
                resampled_annotations = resampled_annotations.astype(int)
                # print(len(resampled_annotations))
                # for i in resampled_annotations:
                #     print(i)
                # Detekcja QRS (np. użycie XQRS lub anotacji)
                xqrs = wfdb.processing.XQRS(sig=resampled_signal, fs=fs)
                xqrs.detect()  # Wykrywanie QRS
                self.qrs_indices = xqrs.qrs_inds
                # print(len(self.qrs_indices))

                # Tworzenie segmentów QRS
                for qrs_idx in range(len(self.qrs_indices)):
                    # Znajdź najbliższą etykietę w anotacjach
                    # nearest_label_idx = np.argmin(resampled_annotations - qrs_idx)
                    # lower_values = [val for val in resampled_annotations if val <= qrs_idx]
                    # if lower_values:
                    #     # print(qrs_i
                    #     # print(resampled_annotations)
                    #     # print(lower_values[-1])
                    #     nearest_label_idx = max(lower_values)
                    #     # print(nearest_label_idx)
                    #     # print(len(annotation))
                    #     label = annotation.aux_note[nearest_label_idx]
                    if self.qrs_indices[qrs_idx] >= resampled_annotations[0]:
                        nearest_label_idx = np.argmin(np.abs(resampled_annotations - self.qrs_indices[qrs_idx]))
                        # print(nearest_label_idx)
                        if resampled_annotations[nearest_label_idx] - self.qrs_indices[qrs_idx] > 0: 
                            nearest_label_idx -= 1
                        label = annotation.aux_note[nearest_label_idx]
                        
                            
                        # Sprawdź, czy to AFIB, czy normalny rytm
                        is_afib = label == '(AFIB'
                        # segment = extract_qrs_segment(resampled_signal, qrs_idx, 40)
                        # print(self.qrs_indices[qrs_idx])
                        # print(self.qrs_indices[qrs_idx - N:qrs_idx + N + 1])
                        # print(len)
                        rr = wfdb.processing.calc_rr(self.qrs_indices[(qrs_idx - N):(qrs_idx + N + 1)], fs=fs, min_rr=None, max_rr=None, qrs_units='samples', rr_units='seconds')
                        self.rr_segments.append(rr)
                        # print(len(rr))
                        self.labels.append(1 if is_afib else 0)

    def __len__(self):
        return len(self.rr_segments)

    def __getitem__(self, idx):
        segment = torch.Tensor(self.rr_segments[idx]).unsqueeze(0)  # Dodaj wymiar kanału
        label = self.labels[idx]
        return segment, label

In [30]:
ds = SHDB_QRS(20,'/home/lsriw/Documents/JZ_AUT/dupa/dnn_ecg/shdb/',fs=100)


Przetwarzanie: 037


OverflowError: Python integer 256 out of bounds for uint8

In [18]:
# import matplotlib.pyplot as plt
# plt.plot(ds.qrs_indices[:10], [1 for i in range(10)])
for i in ds.qrs_indices:
    print(i)
# print(ds.qrs_indices)

43
95
141
193
234
274
314
350
389
428
472
517
555
594
634
674
718
759
799
839
879
920
961
1003
1045
1085
1126
1167
1208
1248
1294
1336
1373
1411
1450
1509
1556
1598
1639
1682
1728
1777
1822
1868
1922
1963
2011
2054
2097
2135
2182
2225
2281
2327
2385
2445
2502
2545
2584
2624
2665
2707
2750
2798
2838
2880
2921
2962
3003
3048
3091
3140
3172
3221
3265
3312
3362
3412
3458
3504
3551
3598
3642
3687
3738
3782
3826
3871
3915
3957
4001
4039
4085
4137
4183
4231
4275
4321
4366
4416
4465
4519
4567
4618
4667
4710
4750
4795
4833
4878
4923
4971
5018
5057
5096
5143
5194
5234
5276
5319
5361
5403
5444
5487
5528
5571
5614
5656
5700
5742
5782
5824
5867
5905
5945
5986
6032
6073
6112
6151
6189
6229
6271
6316
6369
6411
6452
6493
6536
6579
6621
6659
6699
6739
6783
6827
6867
6909
6948
6989
7034
7076
7115
7157
7198
7238
7281
7323
7363
7406
7450
7492
7533
7573
7617
7667
7709
7747
7787
7829
7870
7913
7955
7999
8042
8086
8128
8169
8211
8252
8298
8338
8378
8416
8457
8497
8537
8580
8625
8666
8707
8748
8789
8826
8868


In [ ]:
print()

In [4]:
torch.save({'segment': ds[0], 'label': ds[1]}, 'dataset.pt')

# Odczyt z pliku
data = torch.load('dataset.pt')
segment = data['segment']
label = data['label']

/tmp/ipykernel_362598/1111217716.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load('dataset.pt')


In [69]:
# def custom_collate_fn(batch):
#     # Usuń puste próbki z batcha
#     batch = [item for item in batch if item[0].numel() > 0]
#     return torch.utils.data.dataloader.default_collate(batch)
# def custom_collate_fn(batch):
#     # Usuń puste elementy (np. gdy obraz lub etykieta jest pusty)
#     batch = [item for item in batch if item[0].numel() > 0]
#     if len(batch) == 0:
#         # Jeśli cały batch jest pusty, zwróć puste tensory
#         return torch.empty(0), torch.empty(0)
#     # Standardowe łączenie batchy
#     return torch.utils.data.dataloader.default_collate(batch)
def custom_collate_fn(batch):
    # Pobierz maksymalny rozmiar w batchu
    max_length = max(item[0].shape[-1] for item in batch)
    
    # Dodaj padding do próbek
    padded_batch = []
    for data, label in batch:
        padding = (0, max_length - data.shape[-1])  # (left, right)
        padded_data = torch.nn.functional.pad(data, padding, mode='constant', value=0)
        padded_batch.append((padded_data, label))
    
    # Stwórz tensor z batcha
    data, labels = zip(*padded_batch)
    return torch.stack(data), torch.tensor(labels)


In [79]:
print(ds[4][0].size())

torch.Size([1, 40])


In [86]:
tab = []
for i in ds:
    # print(i[1])
    if i[0].size() == torch.Size([1, 40]):
        # print(i[1])
        tab.append(i)

In [87]:
# print(tab[4])
print(len(tab))

166356


In [91]:
print(tab[4])

(tensor([[0.7000, 0.7000, 0.7000, 0.7000, 0.7000, 0.6900, 0.6800, 0.6700, 0.6800,
         0.6900, 0.6800, 0.7000, 0.7000, 0.7200, 0.7200, 0.7200, 0.7400, 0.7300,
         0.7400, 0.7300, 0.7200, 0.7200, 0.7200, 0.7200, 0.7100, 0.7000, 0.7000,
         0.6900, 0.6900, 0.6800, 0.6900, 0.6800, 0.6900, 0.7000, 0.7100, 0.7100,
         0.7000, 0.7100, 0.7000, 0.7100]]), 0)


In [88]:
from torch.utils.data import DataLoader, random_split

train_set, val_set = random_split(tab, [0.8, 0.2])
train = DataLoader(train_set, batch_size=32, shuffle=True,  collate_fn=custom_collate_fn)
val = DataLoader(val_set, batch_size=32, shuffle=True,  collate_fn=custom_collate_fn)

In [57]:
print(train_set[156780])
# print(tab)
print(ds)

(tensor([], size=(1, 0)), 0)


## Model a La resnet

In [107]:

from torchvision.io import read_image
import torch.nn as nn
import torch
from tqdm import tqdm

# class SimpleConv(nn.Module):
#     # def __init__(self, input = 41, input_ch = 1, num_classes = 2):
#     #     super(SimpleConv, self).__init__()
#     #     self.model = nn.Sequential(
#     #         nn.Conv1d(input_ch, 64, kernel_size=6, padding='same'),
#     #         nn.BatchNorm1d(64),
#     #         nn.ReLU(),
#     #         nn.Conv1d(64, 64, kernel_size=3, padding='same'),
#     #         nn.BatchNorm1d(64),
#     #         nn.ReLU(),
#     #         nn.Conv1d(64, 128, kernel_size=3, padding='same'),      # out 1 x 128 x n
#     #         nn.BatchNorm1d(128),
#     #         nn.ReLU(),
#     #         nn.MaxPool1d(2),                        # out 1 x 128 x n//2
#     #         nn.Conv1d(128, 128, kernel_size=3, padding='same'),
#     #         nn.BatchNorm1d(128),
#     #         nn.ReLU(),
#     #         nn.Conv1d(128, 256, kernel_size=3, padding='same'),     # out 1 x 256 x n//2
#     #         nn.BatchNorm1d(256),
#     #         nn.ReLU(),
#     #         nn.MaxPool1d(2),                        # out 1 x 256 x n//4
#     #         nn.Conv1d(256, 256, kernel_size=3, padding='same'),     
#     #         nn.BatchNorm1d(256),
#     #         nn.ReLU(),
#     #         nn.Conv1d(256, 512, kernel_size=3, padding='same'),     
#     #         nn.BatchNorm1d(512),
#     #         nn.ReLU(),
#     #         nn.MaxPool1d(2),                        # out 1 x 512 x n//8
#     #         nn.Flatten(),
#     #         nn.Linear(512*(input//8), 256),
#     #         nn.Dropout(0.5),
#     #         nn.Linear(256, num_classes),
#     #     )
#     #     self.model.to('cuda:0')
#     # def __init__(self, input = 41, input_ch = 1, num_classes = 2):
#     #     super(SimpleConv, self).__init__()
#     #     self.model = nn.Sequential(
#     #         nn.Conv1d(input_ch, 64, kernel_size=6, padding='same'),
#     #         nn.BatchNorm1d(64),
#     #         nn.ReLU(),
#     #         nn.Conv1d(64, 64, kernel_size=3, padding='same'),
#     #         nn.BatchNorm1d(64),
#     #         nn.ReLU(),
#     #         nn.Conv1d(64, 128, kernel_size=3, padding='same'),      # out 1 x 128 x n
#     #         nn.BatchNorm1d(128),
#     #         nn.ReLU(),
#     #         nn.MaxPool1d(2),                        # out 1 x 128 x n//2
#     #         nn.Conv1d(128, 128, kernel_size=3, padding='same'),
#     #         nn.BatchNorm1d(128),
#     #         nn.ReLU(),
#     #         nn.Conv1d(128, 256, kernel_size=3, padding='same'),     # out 1 x 256 x n//2
#     #         nn.BatchNorm1d(256),
#     #         nn.ReLU(),
#     #         nn.MaxPool1d(2),                        # out 1 x 256 x n//4
#     #         nn.Conv1d(256, 256, kernel_size=3, padding='same'),     
#     #         nn.BatchNorm1d(256),
#     #         nn.ReLU(),
#     #         nn.Conv1d(256, 512, kernel_size=3, padding='same'),     
#     #         nn.BatchNorm1d(512),
#     #         nn.ReLU(),
#     #         nn.MaxPool1d(2),                        # out 1 x 512 x n//8
#     #         nn.Flatten(),
#     #         nn.Linear(512*2, 256),  # Poprawiony wymiar wejścia do Linear
#     #         nn.Dropout(0.5),
#     #         nn.Linear(256, num_classes),
#     #     )
#         # self.model.to('cuda:0')
#     def __init__(self, input=41, input_ch=1, num_classes=2):
#         super(SimpleConv, self).__init__()
#         self.model = nn.Sequential(
#             nn.Conv1d(input_ch, 64, kernel_size=6, padding='same'),
#             nn.BatchNorm1d(64),
#             nn.ReLU(),
#             nn.Conv1d(64, 64, kernel_size=3, padding='same'),
#             nn.BatchNorm1d(64),
#             nn.ReLU(),
#             nn.Conv1d(64, 128, kernel_size=3, padding='same'),
#             nn.BatchNorm1d(128),
#             nn.ReLU(),
#             nn.MaxPool1d(2),
#             nn.Conv1d(128, 128, kernel_size=3, padding='same'),
#             nn.BatchNorm1d(128),
#             nn.ReLU(),
#             nn.Conv1d(128, 256, kernel_size=3, padding='same'),
#             nn.BatchNorm1d(256),
#             nn.ReLU(),
#             nn.MaxPool1d(2),
#             nn.Conv1d(256, 256, kernel_size=3, padding='same'),
#             nn.BatchNorm1d(256),
#             nn.ReLU(),
#             nn.Conv1d(256, 512, kernel_size=3, padding='same'),
#             nn.BatchNorm1d(512),
#             nn.ReLU(),
#             nn.MaxPool1d(2),
#             nn.Flatten(),
#             # Zmieniamy ten wymiar na odpowiedni
#             nn.Linear(512 * 5, 256),  # Zakładając, że długość wejściowa to 40, wynikowy wymiar to 512 * 5
#             nn.Dropout(0.5),
#             nn.Linear(256, num_classes),
#         )
#         self.model.to('cuda:0')

#     # def forward(self, x):

#     #     return self.model(x)
#     def forward(self, x):
#         # for layer in self.model:
#         #     x = layer(x)
#         #     print(f"Output shape after {layer}: {x.shape}")
#         # return x
#         # print(f"Input shape: {x.shape}")  # Zobacz, jaki jest wymiar wejściowy
#         # x = self.model[0](x)  # Conv1d
#         # print(f"After Conv1d: {x.shape}")
#         # x = self.model[1](x)  # BatchNorm1d
#         # x = self.model[2](x)  # ReLU
#         # x = self.model[3](x)  # Conv1d
#         # print(f"After second Conv1d: {x.shape}")
#         # x = self.model[4](x)  # BatchNorm1d
#         # x = self.model[5](x)  # ReLU
#         # x = self.model[6](x)  # Conv1d
#         # print(f"After third Conv1d: {x.shape}")
#         # x = self.model[7](x)  # BatchNorm1d
#         # x = self.model[8](x)  # ReLU
#         # x = self.model[9](x)  # MaxPool1d
#         # print(f"After MaxPool1d: {x.shape}")
#         # x = self.model[10](x)  # Conv1d
#         # print(f"After fourth Conv1d: {x.shape}")
#         # x = self.model[11](x)  # BatchNorm1d
#         # x = self.model[12](x)  # ReLU
#         # x = self.model[13](x)  # Conv1d
#         # print(f"After fifth Conv1d: {x.shape}")
#         # x = self.model[14](x)  # BatchNorm1d
#         # x = self.model[15](x)  # ReLU
#         # x = self.model[16](x)  # MaxPool1d
#         # print(f"After second MaxPool1d: {x.shape}")
#         # x = self.model[17](x)  # Flatten
#         # print(f"After Flatten: {x.shape}")
#         # x = self.model[18](x)  # Linear
#         # print(f"After Linear: {x.shape}")
#         # return x
#         print(f"Input shape: {x.shape}")
#         x = self.model[0](x)  # Conv1d
#         print(f"After Conv1d: {x.shape}")
#         x = self.model[1](x)  # BatchNorm1d
#         x = self.model[2](x)  # ReLU
#         x = self.model[3](x)  # Conv1d
#         print(f"After second Conv1d: {x.shape}")
#         x = self.model[4](x)  # BatchNorm1d
#         x = self.model[5](x)  # ReLU
#         x = self.model[6](x)  # Conv1d
#         print(f"After third Conv1d: {x.shape}")
#         x = self.model[7](x)  # BatchNorm1d
#         x = self.model[8](x)  # ReLU
#         x = self.model[9](x)  # MaxPool1d
#         print(f"After MaxPool1d: {x.shape}")
#         x = self.model[10](x)  # Conv1d
#         print(f"After fourth Conv1d: {x.shape}")
#         x = self.model[11](x)  # BatchNorm1d
#         x = self.model[12](x)  # ReLU
#         x = self.model[13](x)  # Conv1d
#         print(f"After fifth Conv1d: {x.shape}")
#         x = self.model[14](x)  # BatchNorm1d
#         x = self.model[15](x)  # ReLU
#         x = self.model[16](x)  # MaxPool1d
#         print(f"After second MaxPool1d: {x.shape}")
        
#         # Flattening the tensor properly:
#         x = x.view(x.size(0), -1)  # Flatten
#         print(f"After Flatten: {x.shape}")
        
#         x = self.model[18](x)  # Linear
#         print(f"After Linear: {x.shape}")
#         return x
class SimpleConv(nn.Module):
    def __init__(self, input=41, input_ch=1, num_classes=2):
        super(SimpleConv, self).__init__()
        self.model = nn.Sequential(
            nn.Conv1d(input_ch, 64, kernel_size=6, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding='same'),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 128, kernel_size=3, padding='same'),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=3, padding='same'),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(256, 256, kernel_size=3, padding='same'),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 512, kernel_size=3, padding='same'),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            # Zmieniamy ten wymiar na odpowiedni
            nn.Linear(512 * 5, 256),  # Zakładając, że długość wejściowa to 40, wynikowy wymiar to 512 * 5
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
        self.model.to('cuda:0')

    def forward(self, x):
        print(f"Input shape: {x.shape}")
        x = self.model[0](x)  # Conv1d
        print(f"After Conv1d: {x.shape}")
        x = self.model[1](x)  # BatchNorm1d
        x = self.model[2](x)  # ReLU
        x = self.model[3](x)  # Conv1d
        print(f"After second Conv1d: {x.shape}")
        x = self.model[4](x)  # BatchNorm1d
        x = self.model[5](x)  # ReLU
        x = self.model[6](x)  # Conv1d
        print(f"After third Conv1d: {x.shape}")
        x = self.model[7](x)  # BatchNorm1d
        x = self.model[8](x)  # ReLU
        x = self.model[9](x)  # MaxPool1d
        print(f"After MaxPool1d: {x.shape}")
        x = self.model[10](x)  # Conv1d
        print(f"After fourth Conv1d: {x.shape}")
        x = self.model[11](x)  # BatchNorm1d
        x = self.model[12](x)  # ReLU
        x = self.model[13](x)  # Conv1d
        print(f"After fifth Conv1d: {x.shape}")
        x = self.model[14](x)  # BatchNorm1d
        x = self.model[15](x)  # ReLU
        x = self.model[16](x)  # MaxPool1d
        print(f"After second MaxPool1d: {x.shape}")
        
        # Flattening the tensor properly:
        x = x.view(x.size(0), -1)  # Flatten
        print(f"After Flatten: {x.shape}")
        
        x = self.model[18](x)  # Linear
        print(f"After Linear: {x.shape}")
        return x

    
    def train_model(self, train_loader, valid_loader, num_epochs = 5, learning_rate=0.001, save_best = True, save_thr = 0.94):
        best_accuracy = 0.0
        total_step = len(train_loader)
        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.RMSprop(self.parameters(), lr=learning_rate, weight_decay = 0.005, momentum = 0.9)  

        for epoch in range(num_epochs):
            # self.train()
            correct = 0
            total = 0
            for i, (images, labels) in enumerate(tqdm(train_loader)):
                # print(f"Batch {i}: images shape = {images.shape}, labels shape = {labels.shape}")
                if images.size(0) == 0:  # Pominięcie pustego batcha
                    # print(f"Pominięto pusty batch w iteracji {i}")
                    continue
   
                # Move tensors to the configured device
                images = images.float().to("cuda")
                labels = labels.type(torch.LongTensor)
                labels = labels.to("cuda")


                optimizer.zero_grad()

                # Forward pass
                outputs = self.forward(images)
                loss = criterion(outputs, labels)
                # Backward and optimize
                loss.backward()
                
                optimizer.step()

                # accuracy
                _, predicted = torch.max(outputs.data, 1)
                correct += (torch.eq(predicted, labels)).sum().item()
                total += labels.size(0)

                del images, labels, outputs

            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}, Accuracy: {:.4f}'
                            .format(epoch+1, num_epochs, i+1, total_step, loss.item(), (float(correct))/total))


            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Validation
            with torch.no_grad():
                correct = 0
                total = 0
                for images, labels in valid_loader:
                    images = images.float().to("cuda")
                    labels = labels.to("cuda")
                    outputs = self.forward(images)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (torch.eq(predicted, labels)).sum().item()
                    del images, labels, outputs
                if(((100 * correct / total) > best_accuracy) and save_best and ((100 * correct / total) > save_thr)):
                    torch.save(self.state_dict(), "simp_conv_qrs.pt")

                print('Accuracy of the network: {} %'.format( 100 * correct / total))

In [111]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleConv(nn.Module):
    def __init__(self, input_size=40, input_ch=1, num_classes=2):
        super(SimpleConv, self).__init__()

        # Uproszczona wersja modelu
        self.conv1 = nn.Conv1d(input_ch, 64, kernel_size=5, padding='same')
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding='same')
        self.bn2 = nn.BatchNorm1d(128)
        self.pool = nn.MaxPool1d(2)

        # Obliczymy wymiar po warstwach Conv1d i MaxPool1d
        self.fc1_input_size = 128 * (input_size // 2)  # Po 2 max pooling
        self.fc1 = nn.Linear(self.fc1_input_size, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # Pass through conv1
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)

        # Pass through conv2
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)

        # Max pooling
        x = self.pool(x)

        # Flatten the output for the fully connected layers
        x = torch.flatten(x, 1)

        # Fully connected layers
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x
    def train_model(self, train_loader, valid_loader, num_epochs = 5, learning_rate=0.001, save_best = True, save_thr = 0.94):
        best_accuracy = 0.0
        total_step = len(train_loader)
        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.RMSprop(self.parameters(), lr=learning_rate, weight_decay = 0.005, momentum = 0.9)  

        for epoch in range(num_epochs):
            # self.train()
            correct = 0
            total = 0
            for i, (images, labels) in enumerate(tqdm(train_loader)):
                # print(f"Batch {i}: images shape = {images.shape}, labels shape = {labels.shape}")
                if images.size(0) == 0:  # Pominięcie pustego batcha
                    # print(f"Pominięto pusty batch w iteracji {i}")
                    continue
   
                # Move tensors to the configured device
                images = images.float().to("cuda")
                labels = labels.type(torch.LongTensor)
                labels = labels.to("cuda")


                optimizer.zero_grad()

                # Forward pass
                outputs = self.forward(images)
                loss = criterion(outputs, labels)
                # Backward and optimize
                loss.backward()
                
                optimizer.step()

                # accuracy
                _, predicted = torch.max(outputs.data, 1)
                correct += (torch.eq(predicted, labels)).sum().item()
                total += labels.size(0)

                del images, labels, outputs

            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}, Accuracy: {:.4f}'
                            .format(epoch+1, num_epochs, i+1, total_step, loss.item(), (float(correct))/total))


            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Validation
            with torch.no_grad():
                correct = 0
                total = 0
                for images, labels in valid_loader:
                    images = images.float().to("cuda")
                    labels = labels.to("cuda")
                    outputs = self.forward(images)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (torch.eq(predicted, labels)).sum().item()
                    del images, labels, outputs
                if(((100 * correct / total) > best_accuracy) and save_best and ((100 * correct / total) > save_thr)):
                    torch.save(self.state_dict(), "simp_conv_qrs.pt")

                print('Accuracy of the network: {} %'.format( 100 * correct / total))

In [114]:
# model_res.train_model(train,val,num_epochs=90,learning_rate=0.0001,save_best=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleConv().to(device)
model.train_model(train,val,num_epochs=6, save_best=True)

100%|██████████| 4159/4159 [00:08<00:00, 517.15it/s]


Epoch [1/6], Step [4159/4159], Loss: 0.0002, Accuracy: 0.9996
Accuracy of the network: 99.98196627693787 %


100%|██████████| 4159/4159 [00:08<00:00, 498.60it/s]


Epoch [2/6], Step [4159/4159], Loss: 0.0006, Accuracy: 0.9998
Accuracy of the network: 99.98196627693787 %


100%|██████████| 4159/4159 [00:07<00:00, 525.43it/s]


...

Epoch [4/6], Step [4159/4159], Loss: 0.0010, Accuracy: 0.9998
Accuracy of the network: 99.98196627693787 %


100%|██████████| 4159/4159 [00:07<00:00, 578.75it/s]


Epoch [5/6], Step [4159/4159], Loss: 0.0007, Accuracy: 0.9998
Accuracy of the network: 99.98196627693787 %


100%|██████████| 4159/4159 [00:07<00:00, 544.04it/s]


Epoch [6/6], Step [4159/4159], Loss: 0.0006, Accuracy: 0.9998
Accuracy of the network: 99.98196627693787 %
